# 00 · Audit Stage 3 data

Run this before preprocessing. It only inspects files and does not modify the dataset.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

# ============================================================
# Repository
# ============================================================

REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "checkout", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

# ============================================================
# Paths
# ============================================================

DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")

DATA_ROOT = DRIVE_ROOT / "DATASET"
COMMA_ROOT = DATA_ROOT / "comma2k19"
RAW_ROOT = COMMA_ROOT / "raw"
PROCESSED_ROOT = COMMA_ROOT / "processed" / "v1"

MANIFEST_ROOT = DRIVE_ROOT / "manifests" / "stage3" / "v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"

for p in [
    PROCESSED_ROOT,
    MANIFEST_ROOT,
    OUTPUT_ROOT,
    PRETRAINED_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

# ============================================================
# Install project from pyproject.toml
# ============================================================

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "-e",
        str(REPO),
    ],
    check=True,
)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

print("Repository     :", REPO)
print("Branch         :", BRANCH)
print("RAW_ROOT       :", RAW_ROOT)
print("PROCESSED_ROOT :", PROCESSED_ROOT)

Mounted at /content/drive
Repository     : /content/Blackbox-Detection
Branch         : stage3-sangchun
RAW_ROOT       : /content/drive/MyDrive/Blackbox-Detection/DATASET/comma2k19/raw
PROCESSED_ROOT : /content/drive/MyDrive/Blackbox-Detection/DATASET/comma2k19/processed/v1


In [2]:
from blackbox_detection.stage3.comma2k19 import find_archives, discover_segments

archives = find_archives(RAW_ROOT)
print('archives:', len(archives))
for p in archives:
    print(f'{p.name:20s} {p.stat().st_size / 2**30:7.2f} GiB')
assert archives, f'No Chunk zip files found under {RAW_ROOT}'

archives: 1
Chunk_1.zip             8.13 GiB


In [3]:
import zipfile

first = archives[0]
refs = discover_segments(first)
print('first archive:', first.name)
print('segments     :', len(refs))
print('first ref    :', refs[0])

with zipfile.ZipFile(first) as zf:
    names = zf.namelist()
    prefix = refs[0].prefix.lower() + '/'
    sample = [n for n in names if n.lower().startswith(prefix)]
    for n in sample[:80]:
        print(n)

first archive: Chunk_1.zip
segments     : 188
first ref    : SegmentRef(archive=PosixPath('/content/drive/MyDrive/Blackbox-Detection/DATASET/comma2k19/raw/Chunk_1.zip'), prefix='Chunk_1/b0c9d2329ad1606b|2018-07-27--06-03-57/3', route_id='b0c9d2329ad1606b|2018-07-27--06-03-57', segment_id='3', video_member='Chunk_1/b0c9d2329ad1606b|2018-07-27--06-03-57/3/video.hevc')
Chunk_1/b0c9d2329ad1606b|2018-07-27--06-03-57/3/
Chunk_1/b0c9d2329ad1606b|2018-07-27--06-03-57/3/processed_log/
Chunk_1/b0c9d2329ad1606b|2018-07-27--06-03-57/3/processed_log/IMU/
Chunk_1/b0c9d2329ad1606b|2018-07-27--06-03-57/3/processed_log/IMU/gyro/
Chunk_1/b0c9d2329ad1606b|2018-07-27--06-03-57/3/processed_log/IMU/gyro/t
Chunk_1/b0c9d2329ad1606b|2018-07-27--06-03-57/3/processed_log/IMU/gyro/value
Chunk_1/b0c9d2329ad1606b|2018-07-27--06-03-57/3/processed_log/IMU/gyro_uncalibrated/
Chunk_1/b0c9d2329ad1606b|2018-07-27--06-03-57/3/processed_log/IMU/gyro_uncalibrated/t
Chunk_1/b0c9d2329ad1606b|2018-07-27--06-03-57/3/processed_l

Expected: each segment should contain `video.hevc`, `processed_log/...car_speed...`, `...steering_angle...`, IMU files, and a global pose/frame-times array. If those paths look substantially different, stop before running notebook 01.